# DroneCount sweep — policy comparison (ACO vs Raster)

Continuation of `dronecount_sweep_plot_raster.ipynb`. Instead of plotting a single
policy, this notebook loads the per-policy sweep folders produced by
`run_dronecount_sweep.py` and compares them head-to-head.

The harness writes each policy's results into its own folder
(`dronecount_sweep_<RoutingPolicy>`), so an ACO sweep and a Raster sweep coexist.
Both are scored with the **same metric**, using the performance-metric definitions:

- the **ground-truth curve** $G(t)$ is the total number of vehicles present in the
  network at time $t$ (from the simulator's per-road occupancy);
- the **detected-vehicle count** $\widehat{G}(t)$ sums the last-scan counts of all
  *fresh* units, and its trajectory is the **Vehicle Detection Curve (VDC)**;
- the **detection deficit** $\Delta(t) = G(t) - \widehat{G}(t)$ integrated over the
  run is the headline **Vehicle Detection Deficit (VDD)**, in vehicle-seconds.

A *unit* is a graph edge for the stigmergic (ACO) policy and an active grid cell for
the Raster baseline; coverage is the fraction of units currently fresh.

## What this produces

1. **Per-policy overlays** — one VDC + coverage figure per policy.
2. **Scaling comparison** — tracking error and coverage vs drone count, both policies.
3. **Head-to-head at a fixed fleet size** — each policy's VDC against the
   ground-truth curve at one chosen drone count.
4. **Detection curves at multiples of 10 drones** — ACO vs Raster VDCs at
   $N = 10, 20, \dots$
5. **Combined summary table** (including per-run VDD).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), None); assert ROOT is not None, "Run this notebook from within the WUInity repository."; OUTPUT_DIR = ROOT / "Examples/NFDRS4_Behave/Roxborough/_output"

# Discover every sweep folder so you can see what's available before mapping.
discovered = sorted([p for p in OUTPUT_DIR.glob("dronecount_sweep*") if p.is_dir()])
print("Discovered sweep folders:")
for p in discovered:
    n = len(list(p.glob("validation_n*.csv")))
    print(f"  {p.name:38s}  {n:3d} runs")

# Map a friendly label -> sweep folder. Edit this to choose which folders to compare.
# Each entry must point at a folder containing validation_n??.csv files.
POLICIES: dict[str, Path] = {
    "Stigmergic": OUTPUT_DIR / "dronecount_sweep_AcoStigmergic_test8_det_traffic",
    "Raster":     OUTPUT_DIR / "dronecount_sweep_Raster_test8_det_traffic",
    # Zero-deposit ACO (Qs=Qv=Qc=0) == uniform random walk baseline.
    "Uniform walk": OUTPUT_DIR / "dronecount_sweep_AcoStigmergic_Roxborough_global_smoke_drones_aco_random_test8_det_traffic",
}

# Stable colour per policy for all comparison plots.
POLICY_COLORS = {
    "Stigmergic":   "tab:blue",
    "Raster":       "tab:red",
    "Uniform walk": "tab:green",
}
_fallback = plt.get_cmap("tab10")
for i, lbl in enumerate(POLICIES):
    POLICY_COLORS.setdefault(lbl, _fallback(i % 10))

# Validate and drop any that don't exist (with a warning) so the rest still runs.
for lbl, folder in list(POLICIES.items()):
    if not folder.exists() or not list(folder.glob("validation_n*.csv")):
        print(f"WARNING: '{lbl}' -> {folder.name} has no runs; dropping from comparison.")
        POLICIES.pop(lbl)
if not POLICIES:
    raise FileNotFoundError("None of the configured POLICIES folders contain runs.")
print(f"\nComparing: {list(POLICIES)}")

In [ ]:
# Trapezoidal integrator (np.trapezoid on numpy>=2, np.trapz on older).
_trapz = getattr(np, "trapezoid", np.trapz)


def load_sweep(folder: Path) -> tuple[dict[int, pd.DataFrame], pd.DataFrame]:
    """Load all validation_n??.csv from a sweep folder.

    Returns (runs, summary):
      runs    : {drone_count -> DataFrame[sim_time_s, truth, swarm_estimate, coverage_pct]}
                Columns map to the metric definitions: truth = ground-truth curve G(t),
                swarm_estimate = detected-vehicle count G_hat(t) (the VDC).
      summary : per-N headline numbers (VDD, mae, mape, mean/peak coverage).
    """
    files = sorted(folder.glob("validation_n*.csv"), key=lambda p: int(p.stem.split("_n")[1]))
    runs: dict[int, pd.DataFrame] = {}
    for fp in files:
        n = int(fp.stem.split("_n")[1])
        runs[n] = pd.read_csv(fp)

    rows = []
    for n, df in runs.items():
        # Vehicle Detection Deficit: VDD = integral(G) - integral(G_hat) = integral(Delta),
        # in vehicle-seconds. Exactly eq. (VDD): difference of the two curve integrals.
        vdd = float(_trapz(df.truth, df.sim_time_s) - _trapz(df.swarm_estimate, df.sim_time_s))
        mask = df.truth > 1.0
        if mask.sum() == 0:
            mae = mape = float("nan")
        else:
            err = np.abs(df.swarm_estimate[mask] - df.truth[mask])
            mae = float(err.mean())
            mape = float((err / df.truth[mask]).mean() * 100)
        rows.append({
            "drones": n,
            "vdd_vehicle_seconds": vdd,
            "mae_vehicles": mae,
            "mape_pct": mape,
            "mean_coverage_pct": float(df.coverage_pct.mean()),
            "peak_coverage_pct": float(df.coverage_pct.max()),
        })
    summary = pd.DataFrame(rows).sort_values("drones").reset_index(drop=True)
    return runs, summary


data: dict[str, dict] = {}
for lbl, folder in POLICIES.items():
    runs, summary = load_sweep(folder)
    ref = runs[min(runs)]  # ground-truth curve is identical across N within a policy
    data[lbl] = {"runs": runs, "summary": summary, "ref": ref, "folder": folder}
    print(f"{lbl:20s}: {len(runs):3d} runs (N={min(runs)}..{max(runs)}), "
          f"peak G(t)={ref.truth.max():.0f} veh, time {ref.sim_time_s.min():.0f}..{ref.sim_time_s.max():.0f}s")

# Sanity: do the policies share the same ground-truth curve? They must, for a fair
# comparison. Align on sim_time_s (not array index) since runs can have different time
# extents, then measure the relative difference over the overlapping window.
labels = list(data)
if len(labels) >= 2:
    a = data[labels[0]]["ref"][["sim_time_s", "truth"]].rename(columns={"truth": "t0"})
    b = data[labels[1]]["ref"][["sim_time_s", "truth"]].rename(columns={"truth": "t1"})
    mrg = a.merge(b, on="sim_time_s")
    if len(mrg):
        rel = (mrg.t0 - mrg.t1).abs().sum() / max(1.0, mrg.t1.sum())
        span0 = (data[labels[0]]["ref"].sim_time_s.min(), data[labels[0]]["ref"].sim_time_s.max())
        span1 = (data[labels[1]]["ref"].sim_time_s.min(), data[labels[1]]["ref"].sim_time_s.max())
        ok = rel < 0.02 and span0 == span1
        verdict = "OK - same scenario" if ok else (
            "DIFFER - the two sweeps used different SUMO scenarios (evacuation timing / "
            "population / EndDateTime). Re-run one so both share the same traffic before "
            "trusting the comparison.")
        print(f"\nGround-truth check ({labels[0]} vs {labels[1]}):")
        print(f"  time spans: {labels[0]}={span0[0]:.0f}..{span0[1]:.0f}s, "
              f"{labels[1]}={span1[0]:.0f}..{span1[1]:.0f}s")
        print(f"  rel. diff over {len(mrg)} overlapping samples = {rel*100:.2f}%  [{verdict}]")
    else:
        print("\nGround-truth check: no overlapping sim_time samples - scenarios are disjoint in time.")


## 1. Per-policy overlays

One figure per policy: the **Vehicle Detection Curve** $\widehat{G}(t)$ (coloured by
drone count) against the single black **ground-truth curve** $G(t)$ on top, and
coverage (fraction of fresh units) on the bottom. Emitted once for each policy so you
can eyeball them side by side.

In [ ]:
def plot_policy_overlay(label: str, blob: dict):
    """Two standalone figures per policy (one PNG each, so they can be dropped
    into a paper figure individually): VDC overlay and coverage overlay.
    """
    runs = blob["runs"]
    ref = blob["ref"]
    counts = sorted(runs)
    n_min, n_max = counts[0], counts[-1]
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(vmin=n_min, vmax=n_max)
    legend_marks = sorted({n_min, max(n_min, n_max // 4), max(n_min, n_max // 2),
                           max(n_min, (3 * n_max) // 4), n_max})
    safe = label.replace(" ", "_").replace("(", "").replace(")", "")

    # --- Figure 1: Vehicle Detection Curve overlay --------------------------
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(ref.sim_time_s, ref.truth, color="black", linewidth=2.2,
            label="ground-truth curve $G(t)$", zorder=10)
    for n in counts:
        df = runs[n]
        c = cmap(norm(n))
        lbl = f"VDC, drones = {n}" if n in legend_marks else None
        ax.plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=0.9, alpha=0.75, label=lbl)
    ax.set_xlabel("simulation time [s]")
    ax.set_ylabel("vehicles")
    ax.set_title(f"[{label}] Vehicle Detection Curve vs ground-truth curve")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, orientation="vertical", aspect=40, pad=0.02).set_label("drone count")
    out = blob["folder"] / f"overlay_{safe}_vdc.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"saved: {out}")
    plt.show()

    # --- Figure 2: coverage overlay -----------------------------------------
    fig, ax = plt.subplots(figsize=(13, 4))
    for n in counts:
        df = runs[n]
        c = cmap(norm(n))
        ax.plot(df.sim_time_s, df.coverage_pct, color=c, linewidth=0.9, alpha=0.75)
    ax.set_xlabel("simulation time [s]")
    ax.set_ylabel("fresh units (% of $\\mathcal{U}$)")
    ax.set_title(f"[{label}] coverage")
    ax.grid(alpha=0.25)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, orientation="vertical", aspect=40, pad=0.02).set_label("drone count")
    out = blob["folder"] / f"overlay_{safe}_coverage.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"saved: {out}")
    plt.show()


for label, blob in data.items():
    plot_policy_overlay(label, blob)

## 1b. Poster figures (legible from ~6 feet)

Large-format duplicates of the overlays above, sized for a printed poster.

- **Hi-res re-render** — the same four standalone figures (VDC overlay + coverage
  overlay, per policy) redrawn on a larger canvas with bigger fonts, thicker lines, and
  DPI 300. Saved with a `_hires` suffix (originals untouched).
- **Combined VDC (poster)** — both policies' **Vehicle Detection Curve** overlays in one
  **2-wide × 1-high** image that *shares* a single title, x-axis label, y-axis label,
  line legend, and drone-count colourbar across the panels (each panel is tagged by a
  name box). Poster-scale type. Saved as `poster_vdc_sidebyside.png`.

The matching **Scalability comparison** poster figure (tracking error + coverage vs
fleet size, shared title / x-axis / legend) is in Section 2.

In [ ]:
def plot_policy_overlay_hires(label: str, blob: dict, scale: float = 0.75, dpi: int = 600):
    """High-resolution, large-type re-render of ``plot_policy_overlay``.

    Same two standalone figures per policy (VDC overlay + coverage overlay), but on a
    bigger canvas with larger fonts, thicker lines, and higher DPI so the text and
    labels stay legible when the figure is viewed or projected from afar. Saved with a
    ``_hires`` suffix so it does not overwrite the originals.
    """
    runs = blob["runs"]
    ref = blob["ref"]
    counts = sorted(runs)
    n_min, n_max = counts[0], counts[-1]
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(vmin=n_min, vmax=n_max)
    legend_marks = sorted({n_min, max(n_min, n_max // 4), max(n_min, n_max // 2),
                           max(n_min, (3 * n_max) // 4), n_max})
    safe = label.replace(" ", "_").replace("(", "").replace(")", "")

    # Font sizes scaled up together so the whole figure reads from a distance.
    fs_title, fs_label, fs_tick, fs_legend, fs_cbar = 22, 18, 15, 13, 16

    # --- Figure 1: Vehicle Detection Curve overlay --------------------------
    fig, ax = plt.subplots(figsize=(13 * scale, 6 * scale))
    ax.plot(ref.sim_time_s, ref.truth, color="black", linewidth=3.2, zorder=10)
    for n in counts:
        df = runs[n]
        c = cmap(norm(n))
        lbl = f"VDC, drones = {n}" if n in legend_marks else None
        ax.plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=1.6, alpha=0.8)
    ax.set_xlabel("simulation time [s]", fontsize=fs_label)
    ax.set_ylabel("vehicles", fontsize=fs_label)
    ax.set_title(f"[{label}] Vehicle Detection Curve vs ground-truth curve", fontsize=fs_title)
    ax.tick_params(labelsize=fs_tick)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=fs_legend, framealpha=0.9)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, orientation="vertical", aspect=40, pad=0.02)
    cb.set_label("drone count", fontsize=fs_cbar)
    cb.ax.tick_params(labelsize=fs_tick)
    out = blob["folder"] / f"overlay_{safe}_vdc_hires.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print(f"saved: {out}")
    plt.show()

    # --- Figure 2: coverage overlay -----------------------------------------
    fig, ax = plt.subplots(figsize=(13 * scale, 6 * scale))
    for n in counts:
        df = runs[n]
        c = cmap(norm(n))
        ax.plot(df.sim_time_s, df.coverage_pct, color=c, linewidth=1.6, alpha=0.8)
    ax.set_xlabel("simulation time [s]", fontsize=fs_label)
    ax.set_ylabel("fresh units (% of $\\mathcal{U}$)", fontsize=fs_label)
    ax.set_title(f"[{label}] coverage", fontsize=fs_title)
    ax.tick_params(labelsize=fs_tick)
    ax.grid(alpha=0.25)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, orientation="vertical", aspect=40, pad=0.02)
    cb.set_label("drone count", fontsize=fs_cbar)
    cb.ax.tick_params(labelsize=fs_tick)
    out = blob["folder"] / f"overlay_{safe}_coverage_hires.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print(f"saved: {out}")
    plt.show()


for label, blob in data.items():
    plot_policy_overlay_hires(label, blob)

In [ ]:
# Poster-scale combined VDC overlays: every policy's Vehicle Detection Curve overlay in
# a single 2-wide x 1-high figure that SHARES one title, one x-axis label, one y-axis
# label, one line legend, and one drone-count colourbar across both panels (instead of
# repeating them per panel). Fonts are sized for a poster read from ~6 feet away; each
# panel is identified by a name box in its top-right corner.
labels = list(data)
ncols = len(labels)

# Shared y-limit across panels so every curve sits on an identical scale.
y_max = max(blob["ref"].truth.max() for blob in data.values())
for blob in data.values():
    for df in blob["runs"].values():
        y_max = max(y_max, float(df.swarm_estimate.max()))
y_max *= 1.08

# Shared colour normalisation across all policies (union of swept drone counts).
all_counts = sorted({n for blob in data.values() for n in blob["runs"]})
n_min, n_max = all_counts[0], all_counts[-1]
cmap = plt.get_cmap("viridis")
norm = plt.Normalize(vmin=n_min, vmax=n_max)

# Poster type sizes (title / panel-name / axis / tick / legend / colourbar label+tick).
FS_SUP, FS_PANEL, FS_AXIS, FS_TICK, FS_LEG, FS_CBL, FS_CBT = 40, 34, 30, 26, 26, 28, 22

fig, axes = plt.subplots(1, ncols, figsize=(15 * ncols, 10), sharey=True, sharex=True)
if ncols == 1:
    axes = [axes]

handles = leg_labels = None
for ax, label in zip(axes, labels):
    blob = data[label]
    runs = blob["runs"]
    ref = blob["ref"]
    counts = sorted(runs)
    legend_marks = sorted({counts[0], max(counts[0], counts[-1] // 4),
                           max(counts[0], counts[-1] // 2),
                           max(counts[0], (3 * counts[-1]) // 4), counts[-1]})
    ax.plot(ref.sim_time_s, ref.truth, color="black", linewidth=4.0,
            label="ground-truth curve $G(t)$", zorder=10)
    for n in counts:
        df = runs[n]
        c = cmap(norm(n))
        lbl = f"VDC, drones = {n}" if n in legend_marks else None
        ax.plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=2.0, alpha=0.85, label=lbl)
    # Panel name inside the axes (top-right, empty area) so it never touches the shared legend.
    ax.text(0.97, 0.95, label, transform=ax.transAxes, ha="right", va="top",
            fontsize=FS_PANEL, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="0.6", alpha=0.9))
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=0.25)
    if handles is None:  # capture handles once; the legend is identical for every panel.
        handles, leg_labels = ax.get_legend_handles_labels()

axes[0].set_ylim(0, y_max)

# SHARED title, x-axis label, y-axis label, and line legend (one of each for the figure).
fig.suptitle("Vehicle Detection Curve vs ground-truth curve", fontsize=FS_SUP, fontweight="bold", y=0.99)
fig.supxlabel("simulation time [s]", fontsize=FS_AXIS)
fig.supylabel("vehicles", fontsize=FS_AXIS)
fig.legend(handles, leg_labels, loc="upper center", bbox_to_anchor=(0.5, 0.93),
           ncol=len(leg_labels), fontsize=FS_LEG, framealpha=0.9)
fig.subplots_adjust(left=0.06, right=0.92, top=0.82, bottom=0.12, wspace=0.06)

# One shared colourbar on the side for drone-count colouring.
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cb = fig.colorbar(sm, ax=list(axes), orientation="vertical", aspect=40, pad=0.02, fraction=0.03)
cb.set_label("drone count", fontsize=FS_CBL)
cb.ax.tick_params(labelsize=FS_CBT)

out = OUTPUT_DIR / "poster_vdc_sidebyside.png"
fig.savefig(out, dpi=300)
print(f"saved: {out}")
plt.show()

## 2. Scaling comparison

Both policies on shared axes: tracking error (MAPE of the detected-vehicle count
against the ground-truth curve) and mean coverage (fraction of fresh units) as
functions of fleet size. This is the money plot — it shows which policy needs fewer
drones to reach a given accuracy / coverage. (The headline VDD per fleet size is in
the summary table, Section 5.)

In [ ]:
# Scaling comparison split into two standalone PNGs so each can be copied into a
# paper figure independently: tracking-error-vs-N and coverage-vs-N.

# --- Figure 1: tracking error (MAPE) vs fleet size --------------------------
fig, ax = plt.subplots(figsize=(7, 5))
for label, blob in data.items():
    s = blob["summary"]
    c = POLICY_COLORS[label]
    ax.plot(s.drones, s.mape_pct, marker="o", color=c, label=label)
ax.set_xlabel("drone count")
ax.set_ylabel("MAPE (%)")
ax.set_title("Tracking error vs fleet size")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
out = OUTPUT_DIR / "policy_compare_scaling_mape.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

# --- Figure 2: coverage vs fleet size ---------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))
for label, blob in data.items():
    s = blob["summary"]
    c = POLICY_COLORS[label]
    ax.plot(s.drones, s.mean_coverage_pct, marker="o", color=c, label=f"{label} (mean)")
    ax.plot(s.drones, s.peak_coverage_pct, marker="^", color=c, linestyle="--",
            alpha=0.5, label=f"{label} (peak)")
ax.set_xlabel("drone count")
ax.set_ylabel("fresh units (% of $\\mathcal{U}$)")
ax.set_title("Coverage vs fleet size")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
out = OUTPUT_DIR / "policy_compare_scaling_coverage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

In [ ]:
# Poster-scale scalability comparison: tracking error (MAPE) and coverage vs fleet size
# in a single 2-wide x 1-high figure that SHARES one title, one x-axis label, and one
# legend (the two algorithms) across both panels. Fonts are sized for a poster read from
# ~6 feet away. Each panel keeps its own y-axis label because the two metrics have
# different units.
FS_SUP, FS_PANEL, FS_AXIS, FS_TICK, FS_LEG = 40, 32, 30, 26, 28

fig, axes = plt.subplots(1, 2, figsize=(15 * 2, 10), sharex=True)
for label, blob in data.items():
    s = blob["summary"]
    c = POLICY_COLORS[label]
    axes[0].plot(s.drones, s.mape_pct, marker="o", markersize=10, linewidth=3.0, color=c, label=label)
    axes[1].plot(s.drones, s.mean_coverage_pct, marker="o", markersize=10, linewidth=3.0, color=c, label=label)

axes[0].set_title("Tracking error (MAPE)", fontsize=FS_PANEL, fontweight="bold")
axes[0].set_ylabel("MAPE (%)", fontsize=FS_AXIS)
axes[1].set_title("Coverage (fresh units)", fontsize=FS_PANEL, fontweight="bold")
axes[1].set_ylabel("fresh units (% of $\\mathcal{U}$)", fontsize=FS_AXIS)
for ax in axes:
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=0.25)

# SHARED title, x-axis label, and legend (one of each for the figure).
handles, leg_labels = axes[0].get_legend_handles_labels()
fig.suptitle("Scalability comparison between the two algorithms", fontsize=FS_SUP, fontweight="bold", y=0.99)
fig.supxlabel("drone count", fontsize=FS_AXIS)
fig.legend(handles, leg_labels, loc="upper center", bbox_to_anchor=(0.5, 0.925),
           ncol=len(leg_labels), fontsize=FS_LEG, framealpha=0.9)
fig.subplots_adjust(left=0.06, right=0.97, top=0.83, bottom=0.12, wspace=0.14)

out = OUTPUT_DIR / "poster_scalability_comparison.png"
fig.savefig(out, dpi=300)
print(f"saved: {out}")
plt.show()

## 3. Head-to-head at a fixed fleet size

Pick one drone count that both policies ran, and overlay each policy's **Vehicle
Detection Curve** $\widehat{G}(t)$ against its **ground-truth curve** $G(t)$. The gap
between a policy's two lines is its **detection deficit** $\Delta(t)$. `N_COMPARE = None`
auto-selects the largest fleet size common to all policies.

In [ ]:
N_COMPARE: int | None = None  # e.g. 10; None = largest common drone count

common = set.intersection(*[set(blob["runs"]) for blob in data.values()])
if not common:
    raise ValueError(f"No common drone count across policies: "
                     f"{ {k: sorted(v['runs']) for k, v in data.items()} }")
n_cmp = N_COMPARE if (N_COMPARE in common) else max(common)
print(f"Common drone counts: {sorted(common)}; comparing at N = {n_cmp}")

# Split into two standalone PNGs so each panel can be dropped into a paper figure
# independently: the VDC head-to-head, and the coverage head-to-head.

# --- Figure 1: VDC head-to-head ---------------------------------------------
fig, ax = plt.subplots(figsize=(13, 5))
# Plot each policy's OWN ground-truth curve (dotted) plus its VDC (solid). If the two
# policies were run on the same scenario their ground-truth curves coincide; if not,
# the divergence is visible here rather than hidden behind a single shared curve.
for label, blob in data.items():
    df = blob["runs"][n_cmp]
    c = POLICY_COLORS[label]
    ax.plot(df.sim_time_s, df.truth, color=c, linestyle=":", linewidth=1.2,
            alpha=0.6, label=f"{label} — ground-truth curve $G(t)$")
    ax.plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=1.6,
            alpha=0.95, label=f"{label} — VDC $\\widehat{{G}}(t)$")
ax.set_xlabel("simulation time [s]")
ax.set_ylabel("vehicles")
ax.set_title(f"Head-to-head at {n_cmp} drones — Vehicle Detection Curve vs ground-truth curve")
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
out = OUTPUT_DIR / f"policy_compare_headtohead_n{n_cmp:02}_vdc.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

# --- Figure 2: coverage head-to-head ----------------------------------------
fig, ax = plt.subplots(figsize=(13, 4))
for label, blob in data.items():
    df = blob["runs"][n_cmp]
    c = POLICY_COLORS[label]
    ax.plot(df.sim_time_s, df.coverage_pct, color=c, linewidth=1.5, alpha=0.95, label=label)
ax.set_xlabel("simulation time [s]")
ax.set_ylabel("fresh units (% of $\\mathcal{U}$)")
ax.set_title(f"Coverage at {n_cmp} drones")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")
fig.tight_layout()
out = OUTPUT_DIR / f"policy_compare_headtohead_n{n_cmp:02}_coverage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

## 4. Detection curves at multiples of 10 drones

For every fleet size $N$ that is a multiple of 10 and present in both policies, one
panel overlays each policy's **Vehicle Detection Curve** $\widehat{G}(t)$ (solid)
against its **ground-truth curve** $G(t)$ (dotted), with the shaded area being the
**detection deficit** $\Delta(t)$ whose integral is the policy's **VDD**. This shows
directly how the two policies' detection behaviour evolves as the swarm grows.

In [ ]:
# One standalone PNG per fleet-size panel, so each can be copied into a paper figure
# independently. Each panel overlays both policies' VDC against the ground-truth curve
# and shades the detection deficit Delta(t); the panel's VDD is in the legend.
mult10 = sorted(n for n in common if (n % 10 == 0) or (n ==1))
if not mult10:
    mult10 = [max(common)]  # fallback when no multiple of 10 was swept
print(f"Detection curves at N = {mult10}")

for n in mult10:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for label, blob in data.items():
        if n not in blob["runs"]:
            continue
        df = blob["runs"][n]
        c = POLICY_COLORS[label]
        # VDD for this policy at this N = integral(G) - integral(G_hat)  [vehicle-seconds].
        vdd = float(_trapz(df.truth, df.sim_time_s) - _trapz(df.swarm_estimate, df.sim_time_s))
        # ground-truth curve (dotted), VDC (solid), detection deficit (shaded between them).
        ax.plot(df.sim_time_s, df.truth, color=c, linestyle=":", linewidth=1.0, alpha=0.55)
        ax.plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=1.6, alpha=0.95,
                label=f"{label}  (VDD = {vdd/1e3:.0f}k veh·s)")
        ax.fill_between(df.sim_time_s, df.swarm_estimate, df.truth,
                        color=c, alpha=0.10, linewidth=0)
    ax.set_title(f"{n} drones — VDC vs ground-truth curve")
    ax.set_xlabel("simulation time [s]")
    ax.set_ylabel("vehicles")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    out = OUTPUT_DIR / f"policy_compare_detection_curve_n{n:02d}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"saved: {out}")
    plt.show()

## 4b. Stigmergic policy: low vs high swarm size

Side-by-side panels of the ACO **Vehicle Detection Curve** $\widehat{G}(t)$ at a
small fleet ($N=5$) and a large fleet ($N=50$), both against the same
**ground-truth curve** $G(t)$. The shaded region between the two curves is the
**detection deficit** $\Delta(t) = G(t) - \widehat{G}(t)$; its area is the run's
**Vehicle Detection Deficit (VDD)** in vehicle-seconds, printed on each panel.
The visual contrast illustrates how much of the surveillance demand a small swarm
misses relative to a large one.

In [ ]:
# ACO at small vs large swarm: one standalone PNG per fleet size, each with the
# VDC, ground-truth curve, and shaded VDD area annotated in vehicle-seconds.
ACO_LABEL = "Stigmergic"
N_LOW, N_HIGH = 1, 50

aco_runs = data[ACO_LABEL]["runs"]
missing = [n for n in (N_LOW, N_HIGH) if n not in aco_runs]
if missing:
    raise KeyError(
        f"{ACO_LABEL} sweep does not contain N={missing}; "
        f"available drone counts: {sorted(aco_runs)}"
    )

# Share a y-axis upper bound across both panels so the visual contrast between low-N
# (lots of shaded area) and high-N (much less) is faithful — otherwise matplotlib
# auto-scales each panel independently and the difference looks smaller than it is.
y_max = max(aco_runs[n].truth.max() for n in (N_LOW, N_HIGH)) * 1.05

for n, color in [(N_LOW, "tab:orange"), (N_HIGH, "tab:blue")]:
    df = aco_runs[n]
    # VDD = integral(G) - integral(G_hat) = area between the two curves [vehicle-seconds].
    vdd = float(_trapz(df.truth, df.sim_time_s) - _trapz(df.swarm_estimate, df.sim_time_s))

    fig, ax = plt.subplots(figsize=(8, 5))
    # Ground-truth curve in black, VDC in policy colour, shaded deficit between them.
    ax.plot(df.sim_time_s, df.truth, color="black", linewidth=2.0,
            label="ground-truth curve $G(t)$", zorder=10)
    ax.plot(df.sim_time_s, df.swarm_estimate, color=color, linewidth=1.8,
            label=f"VDC $\\widehat{{G}}(t)$ — $N={n}$ drones", zorder=9)
    ax.fill_between(df.sim_time_s, df.swarm_estimate, df.truth,
                    color=color, alpha=0.25, linewidth=0,
                    label="detection deficit $\\Delta(t)$")

    # Annotation: VDD area printed inside the panel.
    ax.text(0.03, 0.97, f"VDD = {vdd:,.0f} veh·s\n      = {vdd/1e3:,.1f} k veh·s",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=color, alpha=0.9))

    ax.set_title(f"ACO stigmergic policy — $N={n}$ drones")
    ax.set_xlabel("simulation time [s]")
    ax.set_ylabel("vehicles")
    ax.set_ylim(0, y_max)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    out = data[ACO_LABEL]["folder"] / f"aco_n{n:02d}_vdc_vdd.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"N={n:2d}: VDD = {vdd:,.1f} veh·s  ({vdd/1e3:,.2f} k veh·s) — saved: {out}")
    plt.show()

## 4c. Marginal VDD reduction per added drone (% change)

How much surveillance debt does the **next** drone actually buy? At each fleet size
$N$ we plot the **marginal reduction in VDD relative to the previous fleet size**,
$$\Delta\%\mathrm{VDD}(N) \;=\; \frac{\mathrm{VDD}(N{-}1) - \mathrm{VDD}(N)}{\mathrm{VDD}(N{-}1)} \times 100\%,$$
i.e. "adding the $N$-th drone shrank the previous swarm's surveillance debt by this
percentage." The curve sits above zero while drones are still useful and falls
toward (or wobbles around) zero once the swarm is saturated. A companion cumulative
panel shows the running **total** percentage of the single-drone baseline debt that
has been eliminated by the time the swarm has grown to $N$ drones,
$$\%\mathrm{reduction}(N) \;=\; \frac{\mathrm{VDD}(1) - \mathrm{VDD}(N)}{\mathrm{VDD}(1)} \times 100\%.$$
Reading the cumulative curve is the cleanest way to see when each policy stops
buying meaningful gain — it grows steeply while drones help and flattens once they
stop.

Both policies are on the same axes for direct comparison. The dotted horizontal line
at $0\%$ marks the "added drone does nothing" boundary.

In [ ]:
# Marginal and cumulative reductions computed on VDD-as-%-of-demand, not on
# absolute veh·s. The %-of-demand normalisation cancels per-run demand-integral
# differences, so if the sweep was assembled from multiple SUMO realisations
# (e.g. N=1..50 and N=51..100 run in separate invocations), batch boundaries no
# longer create artificial step changes in the cumulative curve.
#
# Two standalone PNGs:
#   - marginal_vdd_reduction_pct.png  : Delta(VDD%) per added drone, percentage points
#   - cumulative_vdd_reduction_pct.png: cumulative Delta(VDD%) vs the N=1 baseline

# Build per-policy normalised series from the summary frames already loaded.
# Each run's vdd_pct = 100 * vdd_vehicle_seconds / demand_integral is recomputed
# from the validation CSV itself, so this is robust to mixed-batch sweeps.
_trapz_local = getattr(np, "trapezoid", np.trapz)

marginal_series = {}
for label, blob in data.items():
    s = blob["summary"].sort_values("drones").reset_index(drop=True).copy()
    # Compute each run's demand integral by re-reading its validation CSV.
    demand_int = []
    for n in s.drones:
        df = blob["runs"][int(n)]
        demand_int.append(float(_trapz_local(df.truth, df.sim_time_s)))
    s["demand_int"] = demand_int
    s["vdd_pct"]    = 100.0 * s.vdd_vehicle_seconds / s.demand_int
    # Marginal change in VDD% from the (N-1)-drone swarm to the N-drone swarm
    # (positive = the Nth drone reduced VDD% by this many percentage points).
    s["delta_pp"]   = s.vdd_pct.shift(1) - s.vdd_pct
    # Cumulative reduction in percentage points vs N=1 baseline.
    s["cum_pp"]     = s.vdd_pct.iloc[0] - s.vdd_pct
    marginal_series[label] = s

# --- Figure 1: marginal change in VDD% per added drone ----------------------
fig, ax = plt.subplots(figsize=(9, 5))
for label, s in marginal_series.items():
    c = POLICY_COLORS[label]
    valid = s.dropna(subset=["delta_pp"])
    ax.plot(valid.drones, valid.delta_pp, marker="o", linewidth=1.4, color=c, label=label)
ax.axhline(0.0, color="grey", linewidth=0.8, linestyle=":", label="0 pp (no gain)")
ax.set_xlabel("drone count $N$")
ax.set_ylabel(r"$\Delta$(VDD%) from the $N$-th drone  [percentage points]")
ax.set_title("Marginal VDD reduction (percentage points of demand) per added drone")
ax.grid(alpha=0.25)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
out = OUTPUT_DIR / "marginal_vdd_reduction_pct.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

# --- Figure 2: cumulative reduction in VDD% vs N=1 --------------------------
# Poster-scale fonts, thicker lines, bigger markers and higher DPI so the figure
# stays legible from ~10 feet away (matches the hi-res VDC-overlay styling).
fs_title, fs_label, fs_tick, fs_legend = 30, 26, 22, 22
fig, ax = plt.subplots(figsize=(18, 10))
for label, s in marginal_series.items():
    c = POLICY_COLORS[label]
    ax.plot(s.drones, s.cum_pp, marker="o", markersize=9, linewidth=3.0, color=c, label=label)
ax.set_xlabel("drone count $N$", fontsize=fs_label)
ax.set_ylabel("cumulative VDD reduction vs $N{=}1$ [%]", fontsize=fs_label)
ax.set_title("Scalability comparison between the two algorithms", fontsize=fs_title)
ax.tick_params(labelsize=fs_tick)
ax.grid(alpha=0.25)
ax.legend(fontsize=fs_legend)
fig.tight_layout()
out = OUTPUT_DIR / "cumulative_vdd_reduction_pct.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

# Saturation heuristic on the smoothed marginal series.
ROLL_WIN = 5
SAT_PP = 0.1  # percentage points
print(f"\n=== Saturation (smallest N where smoothed marginal gain <= {SAT_PP} pp) ===")
for label, s in marginal_series.items():
    valid = s.dropna(subset=["delta_pp"]).copy()
    if len(valid) < ROLL_WIN:
        print(f"  {label}: insufficient runs")
        continue
    valid["smoothed"] = valid.delta_pp.rolling(ROLL_WIN, min_periods=1, center=True).mean()
    below = valid[valid.smoothed <= SAT_PP]
    sat = int(below.drones.iloc[0]) if len(below) else None
    peak_pp = valid.smoothed.max()
    peak_n = int(valid.loc[valid.smoothed.idxmax(), "drones"])
    final_cum = float(s.cum_pp.iloc[-1])
    print(f"  {label}: peak smoothed gain {peak_pp:5.2f} pp at N={peak_n:2d}; "
          f"saturates at N={sat if sat else 'never'};  cumulative @ N={int(s.drones.iloc[-1])} = {final_cum:.1f} pp")


## 5. Combined summary table

Per-policy, per-N headline numbers in one frame — including the **Vehicle Detection
Deficit (VDD)** in vehicle-seconds — also saved to `policy_compare_summary.csv`.

In [ ]:
frames = []
for label, blob in data.items():
    s = blob["summary"].copy()
    s.insert(0, "policy", label)
    frames.append(s)
combined = pd.concat(frames, ignore_index=True)
out = OUTPUT_DIR / "policy_compare_summary.csv"
combined.to_csv(out, index=False)
print(f"saved: {out}")

# Headline metric: Vehicle Detection Deficit (vehicle-seconds) by policy across fleet sizes.
pivot_vdd = combined.pivot_table(index="drones", columns="policy", values="vdd_vehicle_seconds")
print("\nVDD (vehicle-seconds) by drone count:")
print(pivot_vdd.round(0).astype("Int64").to_string())

# Tracking error (MAPE) by policy across drone counts.
pivot_mape = combined.pivot_table(index="drones", columns="policy", values="mape_pct")
print("\nMAPE (%) by drone count:")
print(pivot_mape.round(1).to_string())

pivot_cov = combined.pivot_table(index="drones", columns="policy", values="mean_coverage_pct")
print("\nMean coverage (% fresh units) by drone count:")
print(pivot_cov.round(1).to_string())
combined